<a target="_blank" href="https://colab.research.google.com/github/tuliosg/linguistica-computacional/blob/main/notebooks/2-Processando%20textos.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Processando textos**
Neste notebook, vamos entender as etapas envolvidas no processamento de textos, como o pré-processamento, normalização e análise.

### **Dependências**

In [ ]:
import re
from collections import Counter

In [ ]:
!mkdir -p ../content/data

!curl -L -o ../content/data/noticia_sintetica.txt https://raw.githubusercontent.com/tuliosg/linguistica-computacional/refs/heads/main/data/noticia_sintetica.txt
!curl -L -o ../data/ent_sintetica.xlsx https://raw.githubusercontent.com/tuliosg/linguistica-computacional/refs/heads/main/data/ent_sintetica.xlsx

### **Carregando os dados**
Vamos continuar usando os mesmos dados de exemplo do notebook anterior.

In [ ]:
conteudo_txt = open('../data/noticia_sintetica.txt', 'r', encoding='utf-8').read()

## **Pré-processamento**
Na programação, o texto é entendido como uma sequência de caraceteres, ou seja, uma lista de símbolos. E esse conceito é fundamental para entender o funcioamento de algumas técnicas de processamento.

In [ ]:
palavra = "teste"
print(list(palavra))

### **Normalização**
A normalização converte as palavras para alguma forma padrão, reduzindo a variabilidade
do vocabulário. Aqui vamos ver algumas técnicas básicas, como limpeza de texto e padronização.
Lematização e radicalização, que também são formas de normalização, dependem de recursos
linguísticos (léxicos) e vão aparecer no próximo notebook, quando formos usar bibliotecas
prontas para PLN.

#### **Limpeza**
Uma das formas mais simples e úteis de fazer a limpeza de textos é removendo espaços em branco excessivos. Por exemplo: " palavra" é diferente de "palavra" para o nosso código.

In [ ]:
# notem que há espaços em branco no início e no final da frase
frase_inicial = " Esta é uma frase de teste. "

frase_limpa = frase_inicial.strip()

print(f"Antes: '{frase_inicial}'\nDepois: '{frase_limpa}'")

Outro tipo de espaçamento que pode aparecer com muita frequência é a quebra de linha, representada por `\n` no código. Voltemos ao conteúdo da notícia:

In [ ]:
# estamos exibindo apenas os 200 primeiros caracteres do conteúdo
display(conteudo_txt[:200])

Notem que a quebra de linha aparece duplicada. Isso também pode afetar etapas do pré-processamento, como a segmentação (que veremos na próxima seção). Então, a depender do objetivo, é interessante substituir essas ocorrências duplicadas por ocorrências únicas.

In [ ]:
# para substituir as ocorrências duplicadas de quebra de linha por ocorrências únicas, podemos usar o método `replace`
conteudo_txt_limpo = conteudo_txt.replace('\n\n', '\n')
display(conteudo_txt_limpo[:200])

#### **Padronização**
A padronização é mais uma forma de diminuir a quantidade de erros que podem ocorrer no momento do processamento. Existem duas formas principais de padronização: conversão para maiúsculas e conversão para minúsculas. O intuito de ambas é fazer com que ocorrências como "dados" e "Dados" fiquem iguais, transformando-as em "DADOS" ou "dados".

In [ ]:
# para converter o texto para maiusculas, usamos o método`upper`
maiusculas = conteudo_txt_limpo.upper()
print(maiusculas[:200])

In [ ]:
# para converter o texto para minusculas, usamos o método`lower`
minusculas = conteudo_txt_limpo.lower()
print(minusculas[:200])

### **Segmentação**
Existem várias técnicas para a segmentação de textos, desde separações simples, tomando como base espaços em branco entre as palavras, até segmentações baseadas em aprendizado de máquina, que separam sentenças completas prevendo seu início e fim de acordo com o contexto. A escolha da melhor técnica sempre vai depender do objetivo.  
Vamos começar tratando dos métodos mais tradicionais de segmentação: `split` e delimitação de caracteres.

#### **O método `split`**
O `split` pega uma sequência de palavras e as separa de acordo com o argumento que você passar. Por exemplo: `split(".")` vai quebrar o texto partindo dos pontos.

In [ ]:
# o split sem nenhum argumento separa o texto em palavras, 
# considerando os espaços em branco como delimitadores
texto_separado = conteudo_txt.split()
display(texto_separado)

In [ ]:
# o split com o argumento '.' separa o texto em frases,
# considerando o ponto final como delimitador
frases = conteudo_txt.split('.')
display(frases)

Notem que, como o título da notícia não termina com ponto final, ele aparece junto da primeira frase do texto. Outro erro que esse tipo de delimitação poderia causar é a separação de termos como "Dra." ou "sr.", considerando o que vem antes do ponto como uma frase.

In [ ]:
# vamos separar o texto em linhas, considerando a quebra de linha como delimitador
texto_separado = conteudo_txt.split("\n")
display(texto_separado)

Aqui o problema da quebra de linha duplicada fica evidente. Temos várias ocorrências de frases vazias, pois o código, ao encontrar `\n\n` quebra o texto duas vezes, transformando o "meio" dessa ocorrência em uma frase sem conteúdo.

In [ ]:
# vamos fazer a etapa de limpeza antes de segmentarmos para ver a diferença
texto_limpo  = conteudo_txt.replace('\n\n', '\n')
frases = texto_limpo.split('\n')
display(frases)

## **Buscando ocorrências**
A busca por fenômenos é uma das etapas mais importantes quando pensamos em filtrar os dados para a pesquisa, afinal de contas, queremos encontrar apenas o que é relevante para o nosso problema.   
> *É impossível fazermos essa filtragem manualmente?*   
> Não. Quando pensamos em um *corpus* com algumas centenas ou até milhares de palavras, uma pessoa pode dar conta.   
> Porém, quando trabalhamos na escala de dezenas de milhares ou até milhões de palavras, essa tarefa é completamente inviável até mesmo para grandes equipes.

No âmbito desta oficina, quando falamos de busca, estamos falando em procurar dentro do nosso *corpus* as ocorrências que sejam idênticas ou que sigam algum padrão estabelecido por nós. Neste notebook, mais especificamente, trataremos apenas de buscas baseadas no paradigma simbólico de PLN.

### **Buscas exatas**
As buscas exatas funcionam da mesma forma quando procuramos algum termo num documento: o código vai ler todas as palavras e retornar apenas as que são idênticas ao termo buscado.  
Em Python, a forma mais simples de fazer esse tipo de busca é através da biblioteca `re`, que permite tanto buscas exatas quanto a criação de padrões.

In [ ]:
# vamos buscar todas as ocorrências da palavra "dados" no texto
busca = re.findall("dados", conteudo_txt)
display(busca)


A primeira vista pode não parecer tão util, mas notem que com essa busca vocês passam a saber quantas vezes o termo "dados" aparece no texto. Inclusive, é possível exibir essa informação usando a função `len` - ela diz quantos elementos existem em uma determinada lista:

In [ ]:
# o len é uma função que vai contar o número de elementos na lista, ou seja, quantas vezes a palavra "dados" aparece no texto
len(busca)

### **Busca por padrões**
A melhor forma de encontrar ocorrências que sigam algum tipo de padrão é usando expressões regulares (regex). Um regex é uma sequência de caracteres que forma um padrão de busca. E, para criarmos esse padrão de busca, é importante sabermos quais metacaracteres e sequências usar:  


| Caractere | Descrição | Exemplo |
| --- | --- | --- |
| `[]` | Um conjunto de caracteres | `"[a-m]"` |
| `\` | Sinaliza uma sequência especial (também pode ser usado para escapar caracteres especiais) | `"\d"` |
| `.` | Qualquer caractere (exceto o caractere de nova linha) | `"he..o"` |
| `^` | Começa com | `"^hello"` |
| `$` | Termina com | `"planet$"` |
| `*` | Zero ou mais ocorrências | `"he.*o"` |
| `+` | Uma ou mais ocorrências | `"he.+o"` |
| `?` | Zero ou uma ocorrência | `"he.?o"` |
| `{}` | Exatamente o número especificado de ocorrências | `"he.{2}o"` |
| `|` | Um ou outro (Operador OU) | `"falls|stays"` |
| `()` | Captura e agrupa |  |  

<br>

| Caractere | Descrição | Exemplo |
| :--- | :--- | :--- |
| `\A` | Retorna uma correspondência se os caracteres especificados estiverem no início da string | `"\AThe"` |
| `\b` | Retorna uma correspondência onde os caracteres especificados estão no início ou no fim de uma palavra<br>(o "r" no início garante que a string seja tratada como uma "raw string") | `r"\bain"`<br>`r"ain\b"` |
| `\B` | Retorna uma correspondência onde os caracteres especificados estão presentes, mas NÃO no início (ou no fim) de uma palavra<br>(o "r" no início garante que a string seja tratada como uma "raw string") | `r"\Bain"`<br>`r"ain\B"` |
| `\d` | Retorna uma correspondência onde a string contém dígitos (números de 0 a 9) | `"\d"` |
| `\D` | Retorna uma correspondência onde a string NÃO contém dígitos | `"\D"` |
| `\s` | Retorna uma correspondência onde a string contém um caractere de espaço em branco | `"\s"` |
| `\S` | Retorna uma correspondência onde a string NÃO contém um caractere de espaço em branco | `"\S"` |
| `\w` | Retorna uma correspondência onde a string contém qualquer caractere de palavra (caracteres de a a Z, dígitos de 0 a 9 e o caractere de sublinhado `_`) | `"\w"` |
| `\W` | Retorna uma correspondência onde a string NÃO contém nenhum caractere de palavra | `"\W"` |
| `\Z` | Retorna uma correspondência se os caracteres especificados estiverem no final da string | `"Spain\Z"` |

<br>



| Conjunto | Descrição |
| --- | --- |
| `[arn]` | Retorna uma correspondência onde um dos caracteres especificados (`a`, `r` ou `n`) está presente |
| `[a-n]` | Retorna uma correspondência para qualquer caractere em minúsculo, alfabeticamente entre `a` e `n` |
| `[^arn]` | Retorna uma correspondência para qualquer caractere, EXCETO `a`, `r` e `n` |
| `[0123]` | Retorna uma correspondência onde qualquer um dos dígitos especificados (`0`, `1`, `2` ou `3`) está presente |
| `[0-9]` | Retorna uma correspondência para qualquer dígito entre `0` e `9` |
| `[0-5][0-9]` | Retorna uma correspondência para quaisquer números de dois dígitos entre `00` e `59` |
| `[a-zA-Z]` | Retorna uma correspondência para qualquer caractere alfabeticamente entre `a` e `z`, seja em minúsculo OU maiúsculo |
| `[+]` | Dentro de conjuntos, `+`, `*`, `.`, `|`, `()`, `$`, `{}` não têm significado especial, então `[+]` significa: retorna uma correspondência para qualquer caractere `+` na string |

In [ ]:
padrao_mente = r"\w+mente\b"

busca_mente = re.findall(padrao_mente, conteudo_txt)
display(busca_mente)

In [ ]:
padrao_lh = r"\w+lh\w+\b"

busca_lh = re.findall(padrao_lh, conteudo_txt)
display(busca_lh)

### **Adicionando contexto**
Para adicionarmos contexto (palavras antes e depois), é essencial a segmentação do texto. Por enquanto, vamos trabalhar com o `split` mas, mais para frente, veremos técnicas melhores de tokenização.

In [ ]:
tokens_noticia = conteudo_txt.split()
padrao_busca = re.compile(r"\w+lh\w+\b") 

for indice, token in enumerate(tokens_noticia):

    # aqui a gente sempre olha para o token atual
    if padrao_busca.search(token):
        
        # janela de contexto (2 antes e 2 depois)
        inicio_janela = max(0, indice - 2)
        fim_janela = min(len(tokens_noticia), indice + 3)
        
        contexto = tokens_noticia[inicio_janela:fim_janela]

        print(f"Índice da palavra no texto: {indice} | Palavra: '{token}'")
        print(f"Contexto: {contexto}\n")

## **Frequência**
Vimos anteriormente como verificar todas as ocorrências de uma determinada palavra e, consequentemente, como contabilizar sua frequência. Porém, existe outra forma de fazer essa operação, que é mais rápida e permite generalização.

In [ ]:
#notem que antes de fazer a contagem de frequência, é importante 
# fazer algum tipo de normalização e segmentar o texto em palavras
palavras_normalizadas = conteudo_txt_limpo.lower().split()
contador_frequencia = Counter(palavras_normalizadas)

# o most_common(5) retorna as 5 palavras mais frequentes no texto, 
# juntamente com suas contagens
display(contador_frequencia.most_common(5))

<div style="text-align: right; background-color: #f8f9fa; padding: 10px 15px; border-right: 4px solid #404040; border-radius: 5px; margin-top: 15px; display: inline-block; float: right; min-width: 300px;">
    <span style="font-weight: bold; color: #404040; font-size: 0.95em;">Métodos computacionais para linguística</span><br>
    <span style="color: #6c757d; font-size: 0.85em; font-style: italic;">Explorando fenômenos linguísticos</span><br>
    <span style="color: #595959; font-size: 0.8em; margin-top: 5px; display: inline-block;">Desenvolvido por <strong><a href="https://github.com/tuliosg" target="_blank" style="color: #404040; text-decoration: none;">Túlio Gois</a></strong></span>
</div>
<div style="clear: both;"></div> 